# MONAI Classification – DDH vs Normal

This notebook builds a complete 2D image classification pipeline with MONAI and PyTorch to distinguish DDH vs Normal using your `dataset/<size>/{Normal,DDH}` folders. It includes data discovery, stratified splits, transforms, a DenseNet121 model, training with validation, testing, metrics, and saving the best model.

In [1]:
!python -c "import monai" || pip install -q "monai-weekly[pillow, tqdm]"
!python -c "import matplotlib" || pip install -q matplotlib
%matplotlib inline

In [15]:
import os
import shutil
import tempfile
import matplotlib.pyplot as plt
import PIL
import torch
from torch.utils.tensorboard import SummaryWriter
import numpy as np
from sklearn.metrics import classification_report

from monai.apps import download_and_extract
from monai.config import print_config
from monai.data import decollate_batch, DataLoader
from monai.metrics import ROCAUCMetric
from monai.networks.nets import DenseNet121
from monai.transforms import (
    Activations,
    EnsureChannelFirst,
    AsDiscrete,
    Compose,
    LoadImage,
    RandFlip,
    RandRotate,
    RandZoom,
    ScaleIntensity,
)
from monai.utils import set_determinism

print_config()

MONAI version: 1.4.0
Numpy version: 1.26.4
Pytorch version: 2.2.2
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: 46a5272196a6c2590ca2589029eed8e4d56ff008
MONAI __file__: /Users/<username>/Documents/Projects/thesis/monai-env/lib/python3.11/site-packages/monai/__init__.py

Optional dependencies:
Numpy version: 1.26.4
Pytorch version: 2.2.2
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: 46a5272196a6c2590ca2589029eed8e4d56ff008
MONAI __file__: /Users/<username>/Documents/Projects/thesis/monai-env/lib/python3.11/site-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: 0.4.11
ITK version: 5.4.4
Nibabel version: 5.3.2
scikit-image version: 0.25.2
scipy version: 1.16.3
Pillow version: 12.0.0
Tensorboard version: 2.20.0
gdown version: 5.2.0
TorchVision version: 0.17.2
tqdm version: 4.67.1
lmdb version: 1.7.5
psutil version: 7.1.3
pandas version: 2.3.3
einops version: 0.8.1
transformers versi

In [18]:
# Add after Cell 3 (imports and print_config())

# Cell: Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Using device: {device}")

if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🖥️  Using device: cpu


In [16]:
from pathlib import Path
from typing import List, Tuple

# --- Configuration ---
CLASS_NAMES = ["Normal", "DDH"]
PREFERRED_SIZES = [224, 227, 256, 299, 331]
DATASET_ROOT = Path("dataset")

# Auto-pick the first available size that contains both classes
chosen_size = None
for s in PREFERRED_SIZES:
    size_root = DATASET_ROOT / str(s)
    if all((size_root / c).exists() for c in CLASS_NAMES):
        chosen_size = s
        break

if chosen_size is None:
    # Fallback to 331 if structure partially exists
    chosen_size = 331
# Derived config
SIZE_ROOT = DATASET_ROOT / str(chosen_size)
IMAGE_SIZE = chosen_size
BATCH_SIZE = 16
EPOCHS = 10
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

RESULTS_DIR = Path("results") / "monai_runs"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Using dataset at: {SIZE_ROOT}")
print(f"📁 Classes: {CLASS_NAMES}")
print(f"⚙️ Config → IMAGE_SIZE={IMAGE_SIZE}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}")


✅ Using dataset at: dataset/331
📁 Classes: ['Normal', 'DDH']
⚙️ Config → IMAGE_SIZE=331, BATCH_SIZE=16, EPOCHS=10


In [4]:
set_determinism(seed=0)


In [13]:
# ...existing code...
import os
from pathlib import Path
from PIL import Image

# Point to one size folder only
data_dir = Path("../dataset/331")

# Only consider these classes (avoids picking *_with_labels)
class_names = ["Normal", "DDH"]

# Option A: non-recursive listing (files must be directly in each class folder)
def list_files_non_recursive(cdir):
    return [
        str(cdir / f)
        for f in os.listdir(cdir)
        if (cdir / f).is_file()
    ]

# Option B: recursive listing (if you have nested subfolders)
def list_files_recursive(cdir):
    return [
        str(p)
        for p in cdir.rglob("*")
        if p.is_file()
    ]

image_files = []
counts = []
for cname in class_names:
    cdir = data_dir / cname
    if not cdir.exists():
        raise FileNotFoundError(f"Missing class folder: {cdir}")
    files = list_files_non_recursive(cdir)  # switch to list_files_recursive(cdir) if needed
    image_files.append(files)
    counts.append(len(files))

num_class = len(class_names)
image_files_list = []
image_class = []
for i in range(num_class):
    image_files_list.extend(image_files[i])
    image_class.extend([i] * counts[i])

if not image_files_list:
    raise RuntimeError("No images found. Check data_dir, class folders, and file extensions.")

# Open first valid image to report dimensions
with Image.open(image_files_list[0]) as im:
    image_width, image_height = im.size

num_total = len(image_class)
print(f"Total image count: {num_total}")
print(f"Image dimensions: {image_width} x {image_height}")
print(f"Label names: {class_names}")
print(f"Label counts: {counts}")
# ...existing code...

Total image count: 314
Image dimensions: 331 x 331
Label names: ['Normal', 'DDH']
Label counts: [233, 81]


In [21]:
# Add these cells after your data discovery cell

# Cell: Stratified train/val/test split
from sklearn.model_selection import train_test_split

# First split: train+val vs test
train_val_files, test_files, train_val_labels, test_labels = train_test_split(
    image_files_list,
    image_class,
    test_size=TEST_SPLIT,
    stratify=image_class,
    random_state=0
)

# Second split: train vs val
train_files, val_files, train_labels, val_labels = train_test_split(
    train_val_files,
    train_val_labels,
    test_size=VAL_SPLIT / (1 - TEST_SPLIT),  # adjust proportion
    stratify=train_val_labels,
    random_state=0
)

print(f"📊 Split summary:")
print(f"   Train: {len(train_files)} ({len([l for l in train_labels if l==0])} Normal, {len([l for l in train_labels if l==1])} DDH)")
print(f"   Val:   {len(val_files)} ({len([l for l in val_labels if l==0])} Normal, {len([l for l in val_labels if l==1])} DDH)")
print(f"   Test:  {len(test_files)} ({len([l for l in test_labels if l==0])} Normal, {len([l for l in test_labels if l==1])} DDH)")

# Cell: MONAI Transforms (preprocessing + augmentation)
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityd,
    RandFlipd,
    RandRotated,
    RandZoomd,
    Resized,
    ToTensord,
    Lambdad,
)

# Training transforms: preprocessing + augmentation
train_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    Lambdad(keys=["image"], func=lambda x: x[:1, :, :] if x.shape[0] == 3 else x),  # Convert RGB to grayscale by taking first channel
    ScaleIntensityd(keys=["image"]),  # normalize to [0,1]
    Resized(keys=["image"], spatial_size=(IMAGE_SIZE, IMAGE_SIZE)),
    RandFlipd(keys=["image"], prob=0.5, spatial_axis=0),  # horizontal flip
    RandRotated(keys=["image"], range_x=0.26, prob=0.5, keep_size=True),  # ±15 degrees
    RandZoomd(keys=["image"], min_zoom=0.9, max_zoom=1.1, prob=0.5),
    ToTensord(keys=["image"]),
])

# Validation/test transforms: only preprocessing (no augmentation)
val_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    Lambdad(keys=["image"], func=lambda x: x[:1, :, :] if x.shape[0] == 3 else x),  # Convert RGB to grayscale
    ScaleIntensityd(keys=["image"]),
    Resized(keys=["image"], spatial_size=(IMAGE_SIZE, IMAGE_SIZE)),
    ToTensord(keys=["image"]),
])

print("✅ Transforms defined")

# Cell: Create MONAI datasets
from monai.data import Dataset, DataLoader

# Convert lists to dicts for MONAI
train_dicts = [{"image": img, "label": lbl} for img, lbl in zip(train_files, train_labels)]
val_dicts = [{"image": img, "label": lbl} for img, lbl in zip(val_files, val_labels)]
test_dicts = [{"image": img, "label": lbl} for img, lbl in zip(test_files, test_labels)]

train_ds = Dataset(data=train_dicts, transform=train_transforms)
val_ds = Dataset(data=val_dicts, transform=val_transforms)
test_ds = Dataset(data=test_dicts, transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"✅ DataLoaders ready: {len(train_loader)} train batches, {len(val_loader)} val batches, {len(test_loader)} test batches")

# Cell: Calculate class weights for imbalanced data
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print(f"⚖️ Class weights: Normal={class_weights[0]:.3f}, DDH={class_weights[1]:.3f}")

# Cell: Define ResNet50 model
from monai.networks.nets import resnet50

model = resnet50(
    pretrained=False,
    spatial_dims=2,
    n_input_channels=1,  # grayscale X-ray
    num_classes=2,
).to(device)

print(f"🧠 Model: ResNet50 with {sum(p.numel() for p in model.parameters()):,} parameters")

# Cell: Loss, optimizer, scheduler
loss_function = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("✅ Loss, optimizer, scheduler ready")

# Cell: Training loop with validation
from tqdm import tqdm

best_val_loss = float('inf')
best_model_path = RESULTS_DIR / "best_resnet50.pth"
history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*60}")
    
    # --- Training ---
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch in tqdm(train_loader, desc="Training"):
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
    
    train_loss /= len(train_loader)
    train_acc = 100 * train_correct / train_total
    
    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            
            outputs = model(images)
            loss = loss_function(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_loss /= len(val_loader)
    val_acc = 100 * val_correct / val_total
    
    # Update learning rate
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"✅ Best model saved (val_loss={val_loss:.4f})")
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | LR: {current_lr:.6f}")

print("\n🎉 Training complete!")

# Cell: Plot training history
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax1.plot(history["train_loss"], label="Train Loss", marker='o')
ax1.plot(history["val_loss"], label="Val Loss", marker='s')
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(history["val_acc"], label="Val Accuracy", marker='s', color='green')
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Validation Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_history.png", dpi=150, bbox_inches='tight')
plt.show()

# Cell: Evaluate on test set
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_loss = 0.0
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        
        outputs = model(images)
        loss = loss_function(outputs, labels)
        
        test_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_loss /= len(test_loader)
test_acc = 100 * sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)

print(f"\n{'='*60}")
print(f"TEST RESULTS")
print(f"{'='*60}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

# Cell: Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Test Set')
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Results saved to: {RESULTS_DIR}")

📊 Split summary:
   Train: 219 (162 Normal, 57 DDH)
   Val:   47 (35 Normal, 12 DDH)
   Test:  48 (36 Normal, 12 DDH)
✅ Transforms defined
✅ DataLoaders ready: 14 train batches, 3 val batches, 3 test batches
⚖️ Class weights: Normal=0.676, DDH=1.921
🧠 Model: ResNet50 with 23,509,698 parameters
✅ Loss, optimizer, scheduler ready

Epoch 1/10
🧠 Model: ResNet50 with 23,509,698 parameters
✅ Loss, optimizer, scheduler ready

Epoch 1/10


Training:   0%|          | 0/14 [00:00<?, ?it/s]



PicklingError: Can't pickle <function <lambda> at 0x1427a6520>: attribute lookup <lambda> on __main__ failed